In [1]:
# ── configuration ──────────────────────────────────────────────────────────
DATE = "2025-10-19"   # change to "2026-04-09" etc.
OUTPUT_DIR = "../output"
# ───────────────────────────────────────────────────────────────────────────

In [2]:
import json, cv2, io, numpy as np
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

val_dir = Path(OUTPUT_DIR) / "validation" / "detection" / DATE
manifest = json.loads((val_dir / "manifest.json").read_text())
candidates = manifest["candidates"]

labels_path = val_dir / "labels.json"
if labels_path.exists():
    existing = {e["idx"]: e["label"] for e in json.loads(labels_path.read_text())["labels"]}
else:
    existing = {}

labels = dict(existing)
labeled_count = sum(1 for v in labels.values() if v in ("positive", "negative", "skip"))
print(f"Loaded {len(candidates)} candidates for {DATE}")
print(f"Already labeled: {labeled_count} / {len(candidates)}")

Loaded 36 candidates for 2025-10-19
Already labeled: 30 / 36


In [3]:
def _render_context(cand: dict, size: int = 700) -> np.ndarray:
    ctx_path = val_dir / cand["context_png"]
    img = cv2.imread(str(ctx_path))
    if img is None:
        img = np.zeros((size, size, 3), dtype=np.uint8)
    h, w = img.shape[:2]

    cx_full = int(round(cand["pixel_x"]))
    cy_full = int(round(cand["pixel_y"]))
    half = 400
    tl_x = max(0, cx_full - half)
    tl_y = max(0, cy_full - half)
    dot_x = cx_full - tl_x
    dot_y = cy_full - tl_y

    cv2.circle(img, (dot_x, dot_y), 14, (0, 220, 80), -1)
    cv2.circle(img, (dot_x, dot_y), 14, (255, 255, 255), 2)
    cv2.circle(img, (dot_x, dot_y), 4,  (255, 255, 255), -1)

    arr_len = 90
    ax2 = int(round(dot_x + arr_len * cand["path_dx"]))
    ay2 = int(round(dot_y + arr_len * cand["path_dy"]))
    cv2.arrowedLine(img, (dot_x, dot_y), (ax2, ay2), (0, 220, 80), 2, tipLength=0.22)

    score_txt = f"  score={cand['detection_score']:.3f}" if "detection_score" in cand else ""
    label_txt = f"{cand['callsign']}  {cand['wall_time_utc'][11:19]} UTC{score_txt}"
    cv2.putText(img, label_txt, (dot_x + 18, dot_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(img, label_txt, (dot_x + 18, dot_y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 1, cv2.LINE_AA)

    if w > size:
        scale = size / w
        img = cv2.resize(img, (size, int(round(h * scale))))
    return img


def _render_roi(cand: dict, target_h: int = 220) -> np.ndarray:
    roi_path = val_dir / cand["roi_png"]
    img = cv2.imread(str(roi_path))
    if img is None:
        return np.zeros((target_h, target_h, 3), dtype=np.uint8)
    h, w = img.shape[:2]
    scale = target_h / max(1, h)
    nw = max(1, int(round(w * scale)))
    img = cv2.resize(img, (nw, target_h), interpolation=cv2.INTER_NEAREST)
    cx_roi, cy_roi = nw // 2, target_h // 2
    ax2 = int(round(cx_roi + 50 * cand["path_dx"]))
    ay2 = int(round(cy_roi + 50 * cand["path_dy"]))
    cv2.arrowedLine(img, (cx_roi, cy_roi), (ax2, ay2), (0, 220, 80), 2, tipLength=0.25)
    return img


def _np_to_widget(arr: np.ndarray) -> widgets.Image:
    rgb = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil.save(buf, format="PNG")
    return widgets.Image(value=buf.getvalue(), format="png")

print("Render helpers ready.")

Render helpers ready.


In [4]:
# ── Labelling UI ────────────────────────────────────────────────────────────

# Jump straight to the first unlabeled candidate.
def _first_unlabeled() -> int:
    for i, c in enumerate(candidates):
        if c["idx"] not in labels:
            return i
    return len(candidates) - 1  # all done — stay at the last one

state = {"idx": _first_unlabeled()}

header     = widgets.HTML()
ctx_widget = widgets.Image(format="png")
roi_widget = widgets.Image(format="png")
status_out = widgets.Output()

btn_pos      = widgets.Button(description="✓ Contrail",       button_style="success", layout=widgets.Layout(width="150px", height="44px"))
btn_neg      = widgets.Button(description="✗ No contrail",    button_style="danger",  layout=widgets.Layout(width="150px", height="44px"))
btn_skip_lbl = widgets.Button(description="? Unsure",          button_style="warning", layout=widgets.Layout(width="110px", height="44px"))
btn_prev     = widgets.Button(description="← Back",            button_style="",        layout=widgets.Layout(width="90px",  height="44px"))
btn_next_ul  = widgets.Button(description="⏭ Next unlabeled", button_style="info",    layout=widgets.Layout(width="160px", height="44px"))
btn_save     = widgets.Button(description="💾 Save",           button_style="primary", layout=widgets.Layout(width="100px", height="44px"))
progress     = widgets.IntProgress(min=0, max=len(candidates), description="Progress:",
                                    layout=widgets.Layout(width="420px"))


def _refresh():
    i = state["idx"]
    cand = candidates[i]
    n = len(candidates)
    cur_lbl = labels.get(cand["idx"], "")
    score_html = ""
    if "detection_score" in cand:
        sc = cand["detection_score"]
        col = "#4f4" if sc >= 0.3 else "#fa4" if sc > 0 else "#888"
        score_html = f"&nbsp;&nbsp;<span style='color:{col}'>det={sc:.3f}</span>"
    lbl_col = "green" if cur_lbl == "positive" else "#e44" if cur_lbl == "negative" else "#888"
    lbl_html = f"<b style='color:{lbl_col}'>{cur_lbl or 'unlabeled'}</b>"
    header.value = (
        f"<h3 style='margin:4px 0'>#{i+1}/{n}&nbsp; "
        f"<span style='color:#5af'>{cand['callsign']}</span>&nbsp; "
        f"{cand['wall_time_utc'][0:10]}&nbsp;{cand['wall_time_utc'][11:19]} UTC"
        f"{score_html}&nbsp;&nbsp;{lbl_html}</h3>"
    )
    ctx_widget.value = _np_to_widget(_render_context(cand)).value
    roi_widget.value = _np_to_widget(_render_roi(cand)).value
    progress.value = sum(1 for v in labels.values() if v in ("positive", "negative", "skip"))


def _label(lbl):
    cand = candidates[state["idx"]]
    labels[cand["idx"]] = lbl
    with status_out:
        clear_output(wait=True)
        print(f"  #{state['idx']+1} {cand['callsign']} → {lbl}")
    if state["idx"] < len(candidates) - 1:
        state["idx"] += 1
    _refresh()


def _goto_next_unlabeled(_=None):
    start = state["idx"] + 1
    for i in range(start, len(candidates)):
        if candidates[i]["idx"] not in labels:
            state["idx"] = i
            _refresh()
            return
    # Wrap around from beginning.
    for i in range(0, start):
        if candidates[i]["idx"] not in labels:
            state["idx"] = i
            _refresh()
            return
    with status_out:
        clear_output(wait=True)
        print("All candidates labeled!")


def _save(_=None):
    out = {"labels": [{"idx": k, "label": v} for k, v in sorted(labels.items())]}
    labels_path.write_text(json.dumps(out, indent=2))
    with status_out:
        clear_output(wait=True)
        pos = sum(1 for v in labels.values() if v == "positive")
        neg = sum(1 for v in labels.values() if v == "negative")
        print(f"Saved {len(labels)} labels → {labels_path}  (pos={pos} neg={neg})")


btn_pos.on_click(lambda _: _label("positive"))
btn_neg.on_click(lambda _: _label("negative"))
btn_skip_lbl.on_click(lambda _: _label("skip"))
btn_prev.on_click(lambda _: (state.update({"idx": max(0, state["idx"] - 1)}), _refresh()))
btn_next_ul.on_click(_goto_next_unlabeled)
btn_save.on_click(_save)

ui = widgets.VBox([
    header,
    widgets.HBox([ctx_widget, widgets.VBox([
        widgets.HTML("<b>ROI crop (2×)</b>"),
        roi_widget,
    ])]),
    widgets.HBox([btn_prev, btn_pos, btn_neg, btn_skip_lbl, btn_next_ul, btn_save]),
    progress,
    status_out,
])

_refresh()
display(ui)

In [5]:
# Jump to a specific position by list index (0-based):
# state["idx"] = 12; _refresh()